# Standard ChEBI SFT On Colab

This notebook is the Colab counterpart to the Kaggle SFT notebook, but it stays intentionally thin. It bootstraps the repo in `/content/Thesis`, uses `scripts/init_colab.py` only for dependency installation and ChEBI dataset preparation, then runs `scripts/train_sft.py` directly with the selected repo config.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/mruniverse8/Thesis.git"
REPO_BRANCH = "gflownet"
REPO_DIR = Path("/content/Thesis")

%cd /content
if (REPO_DIR / ".git").exists():
    print(f"Reusing {REPO_DIR}")
elif REPO_DIR.exists():
    raise RuntimeError(f"Existing non-git directory at {REPO_DIR}; delete it and rerun the notebook.")
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", REPO_URL, REPO_BRANCH], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-B", REPO_BRANCH, "FETCH_HEAD"], check=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print({"repo_dir": str(REPO_DIR), "repo_branch": REPO_BRANCH})


In [ ]:
DATASET_MODE = "auto"
CONFIG_OVERRIDE = None
TRAIN_CONFIG = CONFIG_OVERRIDE or Path("configs/sft_chebi20.yaml")

OUTPUT_DIR = REPO_DIR / "outputs" / "chebi20_sft"
BEST_CHECKPOINT = OUTPUT_DIR / "checkpoints" / "best"
RUN_SUMMARY_PATH = OUTPUT_DIR / "run_summary.json"


In [ ]:
%cd {REPO_DIR}
bootstrap_command = [
    sys.executable,
    "scripts/init_colab.py",
    "--stage",
    "sft",
    "--repo-url",
    REPO_URL,
    "--repo-branch",
    REPO_BRANCH,
    "--repo-dir",
    str(REPO_DIR),
    "--dataset-mode",
    DATASET_MODE,
]
if CONFIG_OVERRIDE:
    bootstrap_command.extend(["--config", str(CONFIG_OVERRIDE)])

print("Bootstrapping:", " ".join(str(part) for part in bootstrap_command))
subprocess.run(bootstrap_command, check=True)

training_command = [
    sys.executable,
    "scripts/train_sft.py",
    "--config",
    str(TRAIN_CONFIG),
]
print("Training:", " ".join(str(part) for part in training_command))
subprocess.run(training_command, check=True)


In [ ]:
print({
    "output_dir": str(OUTPUT_DIR),
    "output_dir_exists": OUTPUT_DIR.exists(),
    "best_checkpoint": str(BEST_CHECKPOINT),
    "best_checkpoint_exists": BEST_CHECKPOINT.exists(),
    "run_summary": str(RUN_SUMMARY_PATH),
    "run_summary_exists": RUN_SUMMARY_PATH.exists(),
})
